# Notebook 2: Model Training & Hyperparameter Tuning

Train 10 classifiers with `RandomizedSearchCV` optimized for **ROC-AUC** (5-fold stratified CV).

In [1]:
import json
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.base import clone
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
SPLIT_DIR = ROOT / 'ml_outputs' / 'splits'
MODEL_DIR = ROOT / 'ml_outputs' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

X_train = pd.read_csv(SPLIT_DIR / 'X_train.csv')
X_test = pd.read_csv(SPLIT_DIR / 'X_test.csv')
y_train = pd.read_csv(SPLIT_DIR / 'y_train.csv').squeeze()
y_test = pd.read_csv(SPLIT_DIR / 'y_test.csv').squeeze()
preprocessor = joblib.load(SPLIT_DIR / 'preprocessor.joblib')

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'Train: {len(X_train)} | Test: {len(X_test)} | scale_pos_weight: {scale_pos_weight:.2f}')

Train: 5634 | Test: 1409 | scale_pos_weight: 2.77


In [2]:
def evaluate_model(name, pipeline, X_te, y_te):
    y_pred = pipeline.predict(X_te)
    y_prob = pipeline.predict_proba(X_te)[:, 1]
    return {
        'model': name,
        'accuracy': round(accuracy_score(y_te, y_pred), 4),
        'precision': round(precision_score(y_te, y_pred, zero_division=0), 4),
        'recall': round(recall_score(y_te, y_pred, zero_division=0), 4),
        'f1': round(f1_score(y_te, y_pred, zero_division=0), 4),
        'roc_auc': round(roc_auc_score(y_te, y_prob), 4),
    }

def tune_model(name, pipeline, param_dist, n_iter=12):
    search = RandomizedSearchCV(
        pipeline, param_distributions=param_dist, n_iter=n_iter,
        scoring='roc_auc', cv=CV, random_state=RANDOM_STATE,
        n_jobs=-1, refit=True, verbose=0,
    )
    t0 = time.time()
    search.fit(X_train, y_train)
    elapsed = round(time.time() - t0, 1)
    print(f'{name}: best CV ROC-AUC={search.best_score_:.4f} ({elapsed}s)')
    joblib.dump(search.best_estimator_, MODEL_DIR / f'{name}_best.joblib')
    return search.best_estimator_, search.best_params_, search.best_score_

In [3]:
models_config = {}
tuning_results = {}
results = []

# 1. Logistic Regression
lr_pipe = Pipeline([('prep', clone(preprocessor)), ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE))])
lr_best, lr_params, _ = tune_model('logistic_regression', lr_pipe, {
    'model__C': [0.01, 0.1, 1, 10],
    'model__penalty': ['l2'],
    'model__solver': ['lbfgs', 'liblinear'],
}, n_iter=8)
tuning_results['logistic_regression'] = lr_params
results.append(evaluate_model('logistic_regression', lr_best, X_test, y_test))

logistic_regression: best CV ROC-AUC=0.8636 (12.2s)


In [4]:
# 2. Random Forest
rf_pipe = Pipeline([('prep', clone(preprocessor)), ('model', RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))])
rf_best, rf_params, _ = tune_model('random_forest', rf_pipe, {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [5, 10, 15, None],
    'model__min_samples_split': [2, 5, 10],
}, n_iter=12)
tuning_results['random_forest'] = rf_params
results.append(evaluate_model('random_forest', rf_best, X_test, y_test))

# 3. Gradient Boosting
gb_pipe = Pipeline([('prep', clone(preprocessor)), ('model', GradientBoostingClassifier(random_state=RANDOM_STATE))])
gb_best, gb_params, _ = tune_model('gradient_boosting', gb_pipe, {
    'model__n_estimators': [100, 200],
    'model__max_depth': [3, 5, 7],
    'model__learning_rate': [0.05, 0.1, 0.2],
    'model__subsample': [0.8, 1.0],
}, n_iter=12)
tuning_results['gradient_boosting'] = gb_params
results.append(evaluate_model('gradient_boosting', gb_best, X_test, y_test))

random_forest: best CV ROC-AUC=0.8618 (64.1s)


gradient_boosting: best CV ROC-AUC=0.8632 (169.1s)


In [5]:
# 4. AdaBoost
ada_pipe = Pipeline([('prep', clone(preprocessor)), ('model', AdaBoostClassifier(random_state=RANDOM_STATE))])
ada_best, ada_params, _ = tune_model('adaboost', ada_pipe, {
    'model__n_estimators': [50, 100, 200],
    'model__learning_rate': [0.05, 0.1, 0.5, 1.0],
}, n_iter=10)
tuning_results['adaboost'] = ada_params
results.append(evaluate_model('adaboost', ada_best, X_test, y_test))

# 5. SVM (RBF)
svm_pipe = Pipeline([('prep', clone(preprocessor)), ('model', SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=RANDOM_STATE))])
svm_best, svm_params, _ = tune_model('svm_rbf', svm_pipe, {
    'model__C': [0.1, 1, 10],
    'model__gamma': ['scale', 'auto', 0.01, 0.1],
}, n_iter=10)
tuning_results['svm_rbf'] = svm_params
results.append(evaluate_model('svm_rbf', svm_best, X_test, y_test))

adaboost: best CV ROC-AUC=0.8642 (50.8s)


svm_rbf: best CV ROC-AUC=0.8601 (318.8s)


In [6]:
# 6. KNN
knn_pipe = Pipeline([('prep', clone(preprocessor)), ('model', KNeighborsClassifier())])
knn_best, knn_params, _ = tune_model('knn', knn_pipe, {
    'model__n_neighbors': [3, 5, 7, 11, 15],
    'model__weights': ['uniform', 'distance'],
    'model__p': [1, 2],
}, n_iter=10)
tuning_results['knn'] = knn_params
results.append(evaluate_model('knn', knn_best, X_test, y_test))

# 7. MLP
mlp_pipe = Pipeline([('prep', clone(preprocessor)), ('model', MLPClassifier(max_iter=500, early_stopping=True, random_state=RANDOM_STATE))])
mlp_best, mlp_params, _ = tune_model('mlp', mlp_pipe, {
    'model__hidden_layer_sizes': [(64,), (128,), (64, 32), (128, 64)],
    'model__alpha': [0.0001, 0.001, 0.01],
    'model__learning_rate_init': [0.001, 0.01],
}, n_iter=10)
tuning_results['mlp'] = mlp_params
results.append(evaluate_model('mlp', mlp_best, X_test, y_test))

knn: best CV ROC-AUC=0.8418 (11.2s)


mlp: best CV ROC-AUC=0.8621 (18.8s)


In [7]:
# 8. XGBoost
xgb_pipe = Pipeline([('prep', clone(preprocessor)), ('model', XGBClassifier(
    scale_pos_weight=scale_pos_weight, eval_metric='logloss',
    random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
))])
xgb_best, xgb_params, _ = tune_model('xgboost', xgb_pipe, {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [3, 5, 7],
    'model__learning_rate': [0.05, 0.1, 0.2],
    'model__subsample': [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0],
}, n_iter=12)
tuning_results['xgboost'] = xgb_params
results.append(evaluate_model('xgboost', xgb_best, X_test, y_test))

# 9. LightGBM
lgbm_pipe = Pipeline([('prep', clone(preprocessor)), ('model', LGBMClassifier(
    scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE,
    n_jobs=-1, verbose=-1,
))])
lgbm_best, lgbm_params, _ = tune_model('lightgbm', lgbm_pipe, {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [3, 5, 7, -1],
    'model__learning_rate': [0.05, 0.1, 0.2],
    'model__num_leaves': [15, 31, 63],
    'model__subsample': [0.8, 1.0],
}, n_iter=12)
tuning_results['lightgbm'] = lgbm_params
results.append(evaluate_model('lightgbm', lgbm_best, X_test, y_test))

xgboost: best CV ROC-AUC=0.8643 (22.9s)


lightgbm: best CV ROC-AUC=0.8630 (25.3s)


In [8]:
# 10. CatBoost
cat_pipe = Pipeline([('prep', clone(preprocessor)), ('model', CatBoostClassifier(
    auto_class_weights='Balanced', random_state=RANDOM_STATE, verbose=0,
))])
cat_best, cat_params, _ = tune_model('catboost', cat_pipe, {
    'model__iterations': [100, 200, 300],
    'model__depth': [4, 6, 8],
    'model__learning_rate': [0.05, 0.1, 0.2],
    'model__l2_leaf_reg': [1, 3, 5],
}, n_iter=12)
tuning_results['catboost'] = cat_params
results.append(evaluate_model('catboost', cat_best, X_test, y_test))

comparison = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
comparison.to_csv(ROOT / 'ml_outputs' / 'model_comparison.csv', index=False)

with open(ROOT / 'ml_outputs' / 'tuning_results.json', 'w') as f:
    json.dump({k: {str(pk): pv for pk, pv in v.items()} for k, v in tuning_results.items()}, f, indent=2)

best_name = comparison.iloc[0]['model']
best_model = joblib.load(MODEL_DIR / f'{best_name}_best.joblib')
joblib.dump(best_model, MODEL_DIR / 'best_model.joblib')

print('\n=== Model Comparison (Test Set) ===')
comparison

catboost: best CV ROC-AUC=0.8669 (66.2s)



=== Model Comparison (Test Set) ===


,model,accuracy,precision,recall,f1,roc_auc
9,catboost,0.7580,0.5288,0.8102,0.6399,0.8571
6,mlp,0.8048,0.6542,0.5615,0.6043,0.8558
8,lightgbm,0.7615,0.5336,0.8075,0.6426,0.8558
0,logistic_regression,0.7523,0.5216,0.8075,0.6338,0.8552
1,random_forest,0.7736,0.5551,0.7406,0.6346,0.8543
7,xgboost,0.7587,0.5308,0.7834,0.6328,0.8541
2,gradient_boosting,0.8077,0.6722,0.5374,0.5973,0.8539
4,svm_rbf,0.7402,0.5068,0.7995,0.6203,0.8515
3,adaboost,0.7984,0.6406,0.5481,0.5908,0.8510
5,knn,0.7757,0.5863,0.5267,0.5549,0.8332
